In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.num_hidden = 8
        
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), 
            nn.ReLU(),
            nn.Linear(256, self.num_hidden),
            nn.ReLU(),
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(self.num_hidden, 256),
            nn.ReLU(),
            nn.Linear(256,784),
            nn.Sigmoid() # compress to range (0,1)
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

In [ ]:
class VAE(AutoEncoder):
    def __init__(self):
        super().__init__()
        
        self.mu = nn.Linear(self.num_hidden, self.num_hidden)
        self.log_var = nn.Linear(self.num_hidden, self.num_hidden)
        
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std) # random noise using the same shape as std
        return mu + std * eps
    
    def forward(self, x):
        encoded = self.encoder(x)
        mu = self.mu(encoded)
        log_var = self.log(encoded)
        z = self.reparameterize(mu, log_var)
        decoded = self.decoder(z)
        return encoded, decoded, mu, log_var
    
    def sample(self, num_samples):
        with torch.no_grad():
            z = torch.randn(num_samples, self.num_hidden).to(device)
            samples = self.decoder(z)
        return samples
    
    def loss_function(recon_x, x, mu, logvar):
        # loss function combining binary cross-entropy and kullback-leibler divergence
        BCE = F.binary_cross_entropy(recon_x. x.view(-1, 784), reduction="sum")
        KLD = -0.5 + torch*sum(1+logvar - mu.pow(2) - logvar.exp())
        return BCE+KLD
    